# `run_all` — Train, evaluate and visualise four IV-surface pipelines

This notebook is the **single entry point** for the project. Running it
top-to-bottom will:

1. Build the unified, Moussa-filtered, BS-inverted quote set from the raw
   CSVs in `../data_csv/` (cached to `data/unified/quotes.pkl`).
2. Train each of the four pipelines on the train split and dump
   per-day (12, 11) IV surfaces:
    - **A** ISNN — per-day input-specified neural net on call prices
    - **B** HyperIV — Set-Transformer hyper-network in IV space
    - **C** HyperISNN — our novel residual hyper-network in price space
    - **D** SSVI — Gatheral–Jacquier parametric surface, calibrated daily
3. Run **Phase-1** (same-day fit) metrics on the test set, in IV and
   price space.
4. Train one VolGAN per pipeline on the pipeline's training-day surfaces.
5. Run **Phase-2** (next-day predict) metrics on the test set.
6. Render every figure in `plots/` and write a markdown summary to
   `data/unified/results/summary.md`.

**Caching:** every expensive artefact is written to `data/unified/` and
loaded from disk on subsequent runs. Delete the corresponding pickle (or
the entire `data/unified/` folder) to force a rebuild. There is no manual
step between cells — re-execute and the notebook will pick up where the
caches left off.

> **Quick mode** — if you only want a smoke test (≈30 train + 5 val + 10
> test days, tiny epoch counts), set `QUICK = True` in the configuration
> cell below. Quick-mode caches are written under `data/unified/quick/`
> so they don't clobber the full-run caches.

## 1. Configuration

Toggle `QUICK` for a fast smoke run. Everything else is auto-detected.

In [ ]:
QUICK = False           # True = 30 train + 5 val + 10 test, tiny epochs
PIPELINES = ("A", "B", "C", "D")
VERBOSE = True

import os, sys, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(os.getcwd()).resolve()
assert (PROJECT_ROOT / "code_files").is_dir(), (
    f"Run this notebook from main_project/. cwd={PROJECT_ROOT}"
)
sys.path.insert(0, str(PROJECT_ROOT / "code_files"))

CSV_DIR = (PROJECT_ROOT / ".." / "data_csv").resolve()
PLOTS_DIR = PROJECT_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

print(f"project root : {PROJECT_ROOT}")
print(f"csv dir      : {CSV_DIR}  (exists: {CSV_DIR.is_dir()})")
print(f"quick mode   : {QUICK}")

## 2. Build (or load) the unified quote set

Reads the six CSVs in `../data_csv/`, runs the shared preprocessing,
applies the Moussa real-time arbitrage filter, BS-inverts every cleaned
mid-price into a market IV, then splits dates by year (2013–17 train,
2018 val, 2019 test). The result is cached as a single dict at
`data/unified/quotes.pkl`.

In [ ]:
from shared_eval.unified_dataset import load_unified_dataset

payload = load_unified_dataset(csv_dir=str(CSV_DIR))

splits = payload["splits"]
print(
    f"train: {len(splits['train'])} days   "
    f"val: {len(splits['val'])} days   "
    f"test: {len(splits['test'])} days"
)

d0 = sorted(payload["quotes_dict"])[0]
df0 = payload["quotes_dict"][d0]
print(f"\nSample day {d0.date()} — {len(df0)} quotes")
df0.head()

### 2a. Spot, side info, and a peek at one IV smile

A quick visual sanity check that the data looks like SPX. Below, the
spot path and the realised-vol side-info we feed into VolGAN, plus one
test-day smile.

In [ ]:
side = payload["side_info"]
St = payload["market_data"]["St_df"].iloc[:, 0]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].plot(St.index, St.values, lw=0.8)
axes[0].set_title("SPX spot"); axes[0].set_xlabel("date")
axes[1].plot(side.index, side["rv21_lag1"], lw=0.8)
axes[1].set_title("21-day realised vol (lagged)"); axes[1].set_xlabel("date")
plt.tight_layout(); plt.show()

d_show = splits["test"][0]
df_show = payload["quotes_dict"][pd.Timestamp(d_show).normalize()]
near = df_show.iloc[(df_show["maturity_days"] - 30).abs().argsort()[:60]]
near = near.sort_values("log_moneyness")
plt.figure(figsize=(6, 3))
plt.plot(near["log_moneyness"], near["sigma_market"], ".")
plt.title(f"~30d smile, {pd.Timestamp(d_show).date()}")
plt.xlabel("log-moneyness k"); plt.ylabel(r"$\sigma_{\rm market}$")
plt.tight_layout(); plt.show()

## 3. Train (or load) the four pipelines, then materialise their surfaces

Each pipeline writes its trained state under `data/unified/models/<P>/`
and a dict `{date → (12, 11) IV grid}` under `data/unified/surfaces/<P>.pkl`.
Both are reloaded on subsequent notebook runs, so this cell is
**fast on a warm cache** and **slow on the first run**.

What each pipeline is doing:
- **A — ISNN.** Per-day input-specified network fit on call prices. Strict
  no-arbitrage by construction; one optimisation per day.
- **B — HyperIV.** Trained once on the train split. A Set Transformer
  encodes the day's quotes into a context vector, which a hyper-network
  decodes into the weights of a tiny IV-MLP.
- **C — HyperISNN.** Our novel residual hyper-network: it produces the
  strictly-positive weights of an ISNN-2, in price space.
- **D — SSVI.** L-BFGS-B fit of the Gatheral–Jacquier 5-parameter
  surface, once per day.

In [ ]:
from shared_eval import runner

paths = runner.make_paths(quick=QUICK)
print("Caching to:", paths["root"])

if QUICK:
    payload = {
        **payload,
        "splits": {
            "train": payload["splits"]["train"][:30],
            "val":   payload["splits"]["val"][:5],
            "test":  payload["splits"]["test"][:10],
        },
    }
    print("[quick] using 30 train + 5 val + 10 test days")

model_states, surfaces = {}, {}
for name in PIPELINES:
    print(f"\n=== Pipeline {name}: train/load ===")
    ms = runner.train_or_load_pipeline(
        name, payload, quick=QUICK, paths=paths,
        rebuild=False, verbose=VERBOSE,
    )
    model_states[name] = ms
    print(f"=== Pipeline {name}: materialise (12x11) surfaces ===")
    surfaces[name] = runner.materialise_surfaces(
        name, ms, payload, paths=paths, verbose=VERBOSE, rebuild=False,
    )
    print(f"  {name}: {len(surfaces[name])} surfaces materialised")

## 4. Phase-1 metrics — same-day fit on the test set

For each pipeline we read its continuous σ at the *exact* (k, T) of every
test-day market quote and report RMSE / MAE in IV space, plus RMSE / MAE
in the price domain (native price for A and C; BS-priced IV for B and D).
Results are cached to `data/unified/results/phase1_fit.json` and
`phase1_price.json`.

In [ ]:
phase1 = runner.run_phase1(model_states, payload, paths=paths, verbose=VERBOSE)
phase1_price = runner.run_phase1_price(model_states, payload, paths=paths, verbose=VERBOSE)

rows = []
for name in PIPELINES:
    o   = phase1.get(name, {}).get("overall", {})
    op_ = phase1_price.get(name, {}).get("overall", {})
    rows.append({
        "pipe":         name,
        "IV RMSE":      o.get("rmse", float("nan")),
        "IV MAE":       o.get("mae",  float("nan")),
        "Price RMSE $": op_.get("rmse", float("nan")),
        "Price MAE $":  op_.get("mae",  float("nan")),
        "n quotes":     o.get("n", 0),
    })
pd.DataFrame(rows).round(5)

## 5. Phase-2 metrics — next-day predict via VolGAN

We train a **separate VolGAN per pipeline** on the pipeline's own
training surfaces and side info, then use it to forecast the next-day
(12, 11) grid on every test day. Δ is the gain over a naïve
*persistence* forecast (yesterday's surface).

In [ ]:
volgan_states = {}
for name in PIPELINES:
    print(f"\n=== VolGAN train/load for pipeline {name} ===")
    volgan_states[name] = runner.train_volgan_for(
        name, surfaces[name], payload, paths=paths,
        quick=QUICK, rebuild=False, verbose=VERBOSE,
    )

phase2 = runner.run_phase2(
    surfaces, volgan_states, payload, paths=paths, verbose=VERBOSE,
)

rows = []
for name in PIPELINES:
    o = phase2.get(name, {}).get("overall", {})
    delta = phase2.get(name, {}).get("delta_over_persistence", float("nan"))
    rows.append({
        "pipe":     name,
        "P2 RMSE":  o.get("rmse", float("nan")),
        "P2 MAE":   o.get("mae",  float("nan")),
        "Δ vs persist": delta,
    })
pd.DataFrame(rows).round(5)

## 6. Markdown summary + every figure in `plots/`

`plots.render_all` regenerates every figure used in the report — headline
RMSE bars, the Phase-1↔Phase-2 scatter, bucketed heatmaps, rolling
RMSE, price headline, the four 3-D surface panels, and per-day smile/price
slices. The markdown table that ends up in the report's results section
is at `data/unified/results/summary.md`.

In [ ]:
from shared_eval import plots as plot_mod

smd = runner.write_summary_md(phase1, phase2, paths, phase1_price=phase1_price)
print("Summary markdown:", smd)
print(smd.read_text())

out_dir = plot_mod.render_all(paths, payload)
print(f"\nFigures written to: {out_dir}")
for p in sorted(out_dir.glob("*.png")):
    print(" ", p.name)

### 6a. Inline gallery

In [ ]:
headline_figs = [
    "01_headline_rmse.png",
    "02_p1_vs_p2_scatter.png",
    "06_rolling_rmse.png",
    "07a_price_headline.png",
]
for f in headline_figs:
    p = PLOTS_DIR / f
    if p.exists():
        display(Image.open(p))

In [ ]:
for f in ("09_3d_surfaces_A.png", "09_3d_surfaces_B.png",
          "09_3d_surfaces_C.png", "09_3d_surfaces_D.png"):
    p = PLOTS_DIR / f
    if p.exists():
        display(Image.open(p))

## 7. Reproduce / extend

- **Force a full rebuild of one pipeline** (e.g. C):

  ```python
  ms = runner.train_or_load_pipeline(
      "C", payload, quick=QUICK, paths=paths,
      rebuild=True, verbose=True,
  )
  ```

- **Wipe all caches and start over:** delete `data/unified/` and rerun
  this notebook top-to-bottom.

- **Quick smoke test:** set `QUICK = True` in cell 1; caches are
  isolated under `data/unified/quick/`.